# Data Cleaning

Cleans the raw data (strip whitespace, drop all-NaN rows/columns, normalize yes/no columns, drop duplicate rows) and shows the before/after report. The cleaned csv lives in `assets/datasets/<dataset>/cleaned_data/` from the first run.

In [1]:
import os
import json
from pathlib import Path

import pandas as pd

DS_ID = os.environ.get("DS_ID", "DS1")

root = Path.cwd()
while not (root / "configs" / "config.json").is_file():
    if root == root.parent:
        raise FileNotFoundError("configs/config.json not found above the working directory")
    root = root.parent

config = json.loads((root / "configs" / "config.json").read_text())
ds = config["datasets"]["list"][DS_ID]
base = (root / "configs" / config["paths"]["datasets_dir"] / ds["folder"]).resolve()
file_key = "train" if "train" in ds["files"]["original"] else "data"

df = pd.read_csv(base / ds["files"]["original"][file_key], na_values=ds["na_values"] or None)
target = ds["target"]["column"]

print(f"Dataset: {DS_ID} - {ds['name']}")
print(f"Source:  {ds['source']}")
print(f"Loaded:  {df.shape[0]} rows x {df.shape[1]} cols, target = '{target}'")
df.head()

Dataset: DS1 - UCI Chronic Kidney Disease Dataset
Source:  https://archive.ics.uci.edu/dataset/336/chronic+kidney+disease
Loaded:  400 rows x 25 cols, target = 'class'


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


In [2]:
def strip_strings(df):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].str.strip()
    return df


def drop_constant_rows(df):
    return df.dropna(how="all")


def drop_empty_columns(df):
    return df.dropna(axis=1, how="all")


def normalize_yes_no(df, target=None):
    """Lowercase yes/no style values in object columns that look binary yes/no."""
    df = df.copy()
    for c in df.columns:
        if c == target or not (pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])):
            continue
        vals = set(df[c].dropna().str.lower().unique())
        if vals and vals <= {"yes", "no"}:
            df[c] = df[c].str.lower().map({"yes": "yes", "no": "no"})
    return df


def drop_duplicate_rows(df):
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    if len(df) != before:
        print(f"Removed {before - len(df)} duplicate rows")
    return df


def clean(df, target=None):
    """Strip whitespace, drop all-NaN rows/columns, normalize yes/no, dedup."""
    df = strip_strings(df)
    df = drop_constant_rows(df)
    df = drop_empty_columns(df)
    df = normalize_yes_no(df, target)
    df = drop_duplicate_rows(df)
    return df


def cleaning_report(df_raw, df_clean):
    rows_raw, cols_raw = df_raw.shape
    rows_clean, cols_clean = df_clean.shape
    dup = df_raw.duplicated().sum()
    lines = [
        "CLEANING REPORT",
        f"Rows:    {rows_raw} -> {rows_clean}  ({rows_raw - rows_clean} removed)",
        f"Columns: {cols_raw} -> {cols_clean}  ({cols_raw - cols_clean} removed)",
        f"Duplicate rows removed:    {dup}",
        f"Valid instances after:     {len(df_clean.dropna())} ({len(df_clean.dropna()) / len(df_clean):.2%})",
        f"Instances with any NaN:    {len(df_clean) - len(df_clean.dropna())}",
        f"Missing cells remaining:   {int(df_clean.isna().sum().sum())}",
    ]
    return "\n".join(lines)


df_raw = df.copy()
df = clean(df_raw, target)
print(cleaning_report(df_raw, df))
print()
print(f"Remaining missing per column (top 10):")
print(df.isna().sum().sort_values(ascending=False).head(10).to_string())
print()
print(f"Columns: {list(df.columns)}")
df.head()

CLEANING REPORT
Rows:    400 -> 400  (0 removed)
Columns: 25 -> 25  (0 removed)
Duplicate rows removed:    0
Valid instances after:     158 (39.50%)
Instances with any NaN:    242
Missing cells remaining:   1012

Remaining missing per column (top 10):
rbc     152
rbcc    131
wbcc    106
pot      88
sod      87
pcv      71
pc       65
hemo     52
su       49
sg       47

Columns: ['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'class']


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd
